# PrismRunner Interactive Walkthrough

This notebook lets you interactively test all core features of `prismrunner`:
1. **Hardware, Dtype & Context-Budget Auto-Detection**
2. **Dynamic Pydantic Schema-to-Prompt Compilation**
3. **Pre-Validation JSON Sanitization & Output Parsing**
4. **Token-Aware Sliding-Window Chunking & Deterministic Stitching**
5. **End-to-End `PrismPipeline` Execution (Dry-Run & Live Local LLM)**

In [ ]:
import sys
from pathlib import Path

# Ensure repo root is on sys.path
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "usertests" else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import json
import pandas as pd
from typing import Literal, Optional, List
from pydantic import BaseModel, Field

from prismrunner import (
    DEFAULT_MODEL_NAME,
    PrismPipeline,
    Record,
    TokenAwareChunker,
    DeterministicStitcher,
    UniversalOutputParser,
    compile_schema_to_prompt,
    generate_hardware_advisory,
    get_optimal_device,
    resolve_device_and_dtype,
)

## 1. Hardware & Context Budget Inspection

In [ ]:
device_name, dtype = resolve_device_and_dtype()
device = get_optimal_device()
advisory = generate_hardware_advisory(DEFAULT_MODEL_NAME, context_limit=32768, device=device)

print(f"Active Device : {device_name} | Optimal Dtype: {dtype}")
print(json.dumps(advisory, indent=2))

## 2. Define a Custom Pydantic Schema & Compile to Prompt

In [ ]:
class ClinicalFinding(BaseModel):
    condition: str = Field(description="Diagnosed medical condition or symptom")
    status: Literal["active", "resolved", "suspected"] = Field(description="Clinical status")
    severity: int = Field(ge=1, le=5, description="Severity rating from 1 (mild) to 5 (critical)")
    notes: Optional[str] = Field(default=None, description="Additional clinical context")

compiled = compile_schema_to_prompt(ClinicalFinding)
print(compiled.render_prompt_block(as_array=True))

## 3. Test the Output Parser on Messy LLM Generations

In [ ]:
raw_llm_response = """Sure! Here are the extracted findings:
```json
[
  {"condition": null, "status": null, "severity": null, "notes": null},
  {"condition": "Type 2 Diabetes", "status": "active", "severity": 3, "notes": "A1c at 7.8%",}
]
```
"""

parse_res = UniversalOutputParser.parse(raw_llm_response, model_cls=ClinicalFinding)
print("Success:", parse_res.success)
print("Validated Items:", [item.model_dump() for item in parse_res.items])

## 4. Run `PrismPipeline` on a Pandas DataFrame
Set `LIVE_INFERENCE = True` below to download/load the model weights and run live local inference on your GPU/MPS/CPU.

In [ ]:
LIVE_INFERENCE = False  # Change to True to run live model generation

df = pd.DataFrame({
    "note_id": ["N-01", "N-02"],
    "text": [
        "Patient presents with acute migraine (severity 4/5). Prior seasonal allergies have resolved.",
        "Suspected mild sleep apnea (2/5); referred for overnight polysomnography.",
    ],
})

pipeline = PrismPipeline(model_name=DEFAULT_MODEL_NAME)
results = pipeline.run(
    task="extract",
    input_data=df,
    text_column="text",
    id_column="note_id",
    schema_cls=ClinicalFinding,
    dry_run=not LIVE_INFERENCE,
)

print(json.dumps(results, indent=2, default=str))

## 5. Sliding-Window Chunking & Multi-Chunk Deduplication

In [ ]:
long_text = (
    "Patient history includes active hypertension managed with Lisinopril. "
    "Follow-up examination confirmed hypertension remains active alongside resolved bronchitis. "
    "New onset suspected osteoarthritis noted in the right knee."
)

chunker = TokenAwareChunker(tokenizer=None, chunk_size=18, chunk_overlap=4, strategy="sentence")
chunks = chunker.chunk_record(Record(id="doc_long", index=0, text=long_text))
for c in chunks:
    print(f"Chunk {c.chunk_index + 1}/{c.total_chunks}: {c.text}")